In [0]:
schemas_df = spark.sql("SHOW SCHEMAS IN bankof420")

business_schemas = [
    row.databaseName
    for row in schemas_df.collect()
    if row.databaseName.lower() not in ("information_schema", "sys")
]


In [0]:
from pyspark.sql import Row

rows = []

for schema in business_schemas:
    tables = spark.sql(f"SHOW TABLES IN bankof420.{schema}").collect()
    
    for t in tables:
        table_name = t.tableName
        desc_df = spark.sql(
            f"DESCRIBE TABLE bankof420.{schema}.{table_name}"
        )
        
        for col in desc_df.collect():
            if col.col_name and not col.col_name.startswith("#"):
                rows.append(
                    Row(
                        catalog="bankof420",
                        schema=schema,
                        table=table_name,
                        column=col.col_name,
                        datatype=col.data_type
                    )
                )


In [0]:
final_df = spark.createDataFrame(rows)
final_df.display()
